To calibrate the model, we use the following market instruments: the Euro Stoxx 50 index, ten dividend futures contracts, and one 3M ATM call option. Since we have twelve market prices but only four model parameters, the calibration problem is overdetermined. We therefore do not impose the pricing equations exactly, but instead minimize the unweighted sum of squared pricing errors.

The parameters to be calibrated are $ (\lambda, C^\ast, C_0, \phi)=:\theta.$

First, the model-implied index price is $S_0^{\mathrm{model}}=F_0=\frac{C^\ast}{r}+\frac{C_0-C^\ast}{r+\lambda}.$ The corresponding market target is $S_0^{\mathrm{mkt}} = 394.118.$

Second, using the expression for the dividend futures price derived in 7) we know that the dividend futures price at date 0 is equal to $f=C^{*}(T_1-T_0)+ \frac{C_0-C^{*}}{\lambda}(e^{-\lambda(T_0)}-e^{-\lambda(T_1)})$ 
So the model-implied price of a dividend futures contract with reference interval $[T_0^k,T_1^k]=[0.25+k,1.25+k],\qquad k=0,\dots,9$ 
is $f_k^{\mathrm{model}}(\theta)=C^\ast(T_1^k-T_0^k)+\frac{C_0-C^\ast}{\lambda}\left(e^{-\lambda T_0^k}-e^{-\lambda T_1^k}\right).$
Since $T_1^k-T_0^k=1$, this becomes $f_k^{\mathrm{model}}(\theta)=C^\ast+\frac{C_0-C^\ast}{\lambda}\left(e^{-\lambda(0.25+k)}-e^{-\lambda(1.25+k)}\right).$

The market dividend futures prices are
$$\begin{aligned}
f_0^{\mathrm{mkt}} &= 17.1981,\\
f_1^{\mathrm{mkt}} &= 18.7223,\\
f_2^{\mathrm{mkt}} &= 19.4360,\\
f_3^{\mathrm{mkt}} &= 19.7445,\\
f_4^{\mathrm{mkt}} &= 19.8833,\\
f_5^{\mathrm{mkt}} &= 19.9623,\\
f_6^{\mathrm{mkt}} &= 19.9703,\\
f_7^{\mathrm{mkt}} &= 20.0066,\\
f_8^{\mathrm{mkt}} &= 20.0019,\\
f_9^{\mathrm{mkt}} &= 20.0031.
\end{aligned}$$

Finally, the 3M ATM call option is included as an additional calibration target. The market price of this option is obtained from the Black-Scholes formula using the quoted implied volatility $\widehat{\sigma}=0.04635,$ with maturity $T=3M=0.25$ and strike $K=S_0=394.118.$
Thus, $C_{\mathrm{call}}^{\mathrm{mkt}}=C_{\mathrm{BS}}(S_0,K,T,r,\widehat{\sigma}).$

The model-implied call price is computed from the density of \(S_T\) derived in question 11):
$C_{\mathrm{call}}^{\mathrm{model}}(\theta)=e^{-rT}\int_{\max(K, s_{min})}^\infty (s-K) p_{S_T}(s;\theta)\,ds,$ where  $p_{S_T}(s;\theta)= $ and $s_{min}=\frac{\lambda C^{*}}{r(r+\lambda)}$

We use them to solve a least-squares calibration problem stated as 
$$\min_{\lambda,\,C^\ast,\,C_0,\,\phi}
\sum_{i=1}^{12}
\left(
\text{model price}_i - \text{market price}_i
\right)^2=\min_{\lambda,\,C^\ast,\,C_0,\,\phi}\left(S_0^{\mathrm{model}} - S_0^{\mathrm{mkt}}\right)^2+\sum_{k=0}^{9}\left(f_k^{\mathrm{model}}(\theta) - f_k^{\mathrm{mkt}}\right)^2+\left(C_{\mathrm{call}}^{\mathrm{model}}(\theta-C_{\mathrm{call}}^{\mathrm{mkt}}\right)^2.$$

The optimization is performed subject to the constraints $\lambda>0, C^\ast>0, C_0>0, \phi>0,$ and $\phi^2 < 2\lambda C^\ast.$

In [2]:
import numpy as np
from scipy.stats import norm

def bsm_call(t, s, T, K, r, delta, sigma):
    tau = T - t

    d_plus = (
        np.log(np.exp((r - delta)*tau)*s/ K)
        / (sigma*np.sqrt(tau))
        + 0.5*sigma*np.sqrt(tau)
    )

    d_minus = d_plus - sigma*np.sqrt(tau)

    return (
        np.exp(-delta*tau)*s*norm.cdf(d_plus)
        - np.exp(-r*tau)*K*norm.cdf(d_minus)
    )


S0 = 394.118
K = S0
t = 0.0
T = 0.25
r = 0.05
delta = 0.0
sigma_hat = 0.04635

call_market = bsm_call(t, S0, T, K, r, delta, sigma_hat)

print(call_market)

6.583300428843302


The market price of the 3M ATM call is obtained from the Black--Scholes--Merton formula. 
Let $\tau = T-t$ denote the time to maturity. In the BSM model, the arbitrage price of a European call is
$$
c(t,s)=e^{-\delta \tau}s\Phi\bigl(d_+(\tau,s)\bigr)-e^{-r\tau}K\Phi\bigl(d_-(\tau,s)\bigr),$$
where
$$d_\pm(\tau,s)=\frac{1}{\sigma\sqrt{\tau}}\log\left(\frac{e^{(r-\delta)\tau}s}{K}\right)\pm\frac{\sigma}{2}\sqrt{\tau}.$$

In our calibration, we use $$t=0,\qquad \tau=T=0.25,\qquad s=S_0=394.118, K=S_0=394.118,\qquad r=0.05,\qquad \sigma=\widehat{\sigma}=0.04635.$$
Since no continuous dividend yield is specified for the BSM benchmark, we take $\delta=0.$
Hence the market call price is
$$C_{\mathrm{call}}^{\mathrm{mkt}}=c(0,S_0)=S_0\Phi(d_+)-e^{-rT}K\Phi(d_-),$$
with $$d_\pm=\frac{1}{\widehat{\sigma}\sqrt{T}}\log\left(\frac{e^{rT}S_0}{K}\right)\pm\frac{\widehat{\sigma}}{2}\sqrt{T}.$$
We obtained the market price of the 3M ATM call to be $\approx 6.5833$.

In [3]:
from scipy.integrate import quad
from scipy.optimize import least_squares
from scipy.special import iv # Modified Bessel function of the first kind

S0 = 394.118
K = S0
t = 0.0
T = 0.25
r = 0.05
delta = 0.0
sigma_hat = 0.04635

f_mkt = np.array([
    17.1981, 18.7223, 19.4360, 19.7445, 19.8833,
    19.9623, 19.9703, 20.0066, 20.0019, 20.0031
])

ks = np.arange(10)
T0s = 0.25 + ks
T1s = 1.25 + ks


def index_price(lam, Cstar, C0):
    return Cstar/r + (C0 - Cstar)/(r + lam)


def dividend_futures(lam, Cstar, C0): #f_k{\mathrm{model}}(\theta)=C^{*]+\frac{C_0-C^{*}}{\lambda}\left(e^{-\lambda(T_0)}-e^{-\lambda(T_1)}\right).
    return (Cstar + (C0 - Cstar)/lam*(np.exp(-lam*T0s) - np.exp(-lam*T1s)))

def s_min(lam, Cstar, r):
    return (Cstar*lam)/(r*(r + lam))

def index_density(s, T, r, lam, Cstar, C0, phi):
    c =Cstar + (r + lam)*(s - Cstar/r)

    if c <= 0:
        return 0.0

    q = 2*lam*Cstar/phi**2 - 1

    a = 2*lam/(phi**2*(1 - np.exp(-lam*T)))

    density_c = (a* np.exp(-a*(c + np.exp(-lam*T)*C0))
       *((np.exp(lam*T)*c / C0) ** (q / 2))
       *iv(q, 2*a*np.sqrt(np.exp(-lam*T)*c*C0)))

    density_s = density_c*(r + lam)

    return density_s

def model_call(lam, Cstar, C0, phi):
    lower = max(K, s_min(lam, Cstar, r))

    integral, _ = quad(
        lambda s: (s - K)*index_density(s, T, r, lam, Cstar, C0, phi),
        lower,
        np.inf
    )

    return np.exp(-r*T)*integral


def residuals(params):
    lam, Cstar, C0, phi = params

    res = []

    res.append(index_price(lam, Cstar, C0) - S0)

    res.extend(dividend_futures(lam, Cstar, C0) - f_mkt)

    res.append(model_call(lam, Cstar, C0, phi) - call_market)

    feller = 2*lam*Cstar - phi**2 #to ensure 2*lam*Cstar > phi**2 for the density to be well-defined
    if feller <= 0:
        res.append(1e6*(-feller))
    else:
        res.append(0.0)


    return np.array(res)

bounds = [
    (1e-6, 10.0),   # lambda
    (1e-6, 100.0),  # C*
    (1e-6, 100.0),  # C0
    (1e-6, 50.0)    # phi
]

lower_bounds = np.array([b[0] for b in bounds])
upper_bounds = np.array([b[1] for b in bounds])

initial_guess = np.array([0.5, 20.0, 16.0, 4.0])

result = least_squares(
    residuals,
    initial_guess,
    bounds=(lower_bounds, upper_bounds)
)

lam, Cstar, C0, phi = result.x

print("lambda =", lam)
print("C*     =", Cstar)
print("C0     =", C0)
print("phi    =", phi)
print("sigma  =", phi / (r + lam))

print("Index model:", index_price(lam, Cstar, C0))
print("Index market:", S0)

print("Call model:", model_call(lam, Cstar, C0, phi))
print("Call market:", call_market)

print("Futures model:")
print(dividend_futures(lam, Cstar, C0))

print("Futures market:")
print(f_mkt)
print("if the condition satisfied sould be greater than 0:", 2*lam*Cstar - phi**2)

lambda = 0.505542776315318
C*     = 20.052661123714167
C0     = 16.2612686029713
phi    = 4.502771771149047
sigma  = 8.105175628444027
Index model: 394.22855895171966
Index market: 394.118
Call model: 6.453369116352875
Call market: 6.583300428843302
Futures model:
[17.42996565 18.47070866 19.09846205 19.47710914 19.70550078 19.84326161
 19.92635592 19.97647659 20.00670828 20.02494338]
Futures market:
[17.1981 18.7223 19.436  19.7445 19.8833 19.9623 19.9703 20.0066 20.0019
 20.0031]
if the condition satisfied sould be greater than 0: 2.330928683846878e-06


Let $V_t$ denote the price at time $t$ of the 1Y ATM call, and let $G_t$ denote the price at time $t$ of the 2Y futures on the index.
Under the model, once the condition $S_t=F_t$ is imposed, the index level is a deterministic function of the dividend rate $C_t$
$$S_t
=
\frac{C^\ast}{r}
+
\frac{C_t-C^\ast}{r+\lambda}.$$

Hence both the call price and the futures price can be written as functions of the same state variable $C_t$
$$V_t = V(t,C_t), \qquad G_t = G(t,C_t).$$

The dividend rate evolves under $\mathbb Q$ according to
$$dC_t=\lambda(C^\ast-C_t)\,dt +\phi\sqrt{C_t}\,dB_t^{\mathbb Q}.$$

Applying Itô's formula to $V(t,C_t)$, we obtain
$$dV_t=\frac{\partial V}{\partial t}(t,C_t)\,dt+\frac{\partial V}{\partial C}(t,C_t)\,dC_t+\frac12\frac{\partial^2 V}{\partial C^2}(t,C_t)\,d\langle C\rangle_t.$$
Since $$ d\langle C\rangle_t=\phi^2 C_t\,dt, $$
we get $$ dV_t=\left[\frac{\partial V}{\partial t}+\lambda(C^\ast-C_t)\frac{\partial V}{\partial C}+\frac12\phi^2 C_t\frac{\partial^2 V}{\partial C^2}\right](t,C_t)\,dt+\frac{\partial V}{\partial C}(t,C_t)\phi\sqrt{C_t}\,dB_t^{\mathbb Q}.$$

Similarly, applying Itô's formula to $G(t,C_t)$, we obtain
$$
dG_t
=
\left[
\frac{\partial G}{\partial t}
+
\lambda(C^\ast-C_t)\frac{\partial G}{\partial C}
+
\frac12\phi^2 C_t\frac{\partial^2 G}{\partial C^2}
\right](t,C_t)\,dt
+
\frac{\partial G}{\partial C}(t,C_t)\phi\sqrt{C_t}\,dB_t^{\mathbb Q}.
$$

Consider now a self-financing portfolio made of the riskless asset and $\pi_t$ units of the 2Y futures. The riskless asset has no Brownian exposure, so the Brownian exposure of the portfolio comes only from the futures position
$$
\pi_t\,dG_t
=
\cdots\,dt
+
\pi_t
\frac{\partial G}{\partial C}(t,C_t)
\phi\sqrt{C_t}\,dB_t^{\mathbb Q}.
$$

To replicate the call, the Brownian exposure of the portfolio must match the
Brownian exposure of the call. Therefore,
$$
\frac{\partial V}{\partial C}(t,C_t)\phi\sqrt{C_t}
=
\pi_t
\frac{\partial G}{\partial C}(t,C_t)\phi\sqrt{C_t}.
$$
Cancelling the common factor $\phi\sqrt{C_t}$, we obtain
$$
\pi_t
=
\frac{
\frac{\partial V}{\partial C}(t,C_t)
}{
\frac{\partial G}{\partial C}(t,C_t)
}.
$$

In particular, the initial number of 2Y futures contracts is
$$
\pi_0
=
\frac{
\frac{\partial V_0}{\partial C_0}
}{
\frac{\partial G_0}{\partial C_0}
}.
$$

In [4]:
from scipy.special import ivp  # Modified Bessel function derivative with respect to the order

def dp_dC0_factor(s, T, r, lam, Cstar, C0, phi):
    c = Cstar + (r + lam)*(s - Cstar / r)

    if c <= 0:
        return 0.0

    q = 2*lam*Cstar/phi**2 - 1
    a = 2*lam / (phi**2*(1 - np.exp(-lam*T)))

    z = 2*a*np.sqrt(np.exp(-lam*T)*c*C0)

    ratio = ivp(q, z, 1) / iv(q, z)

    return (-a*np.exp(-lam*T)- q / (2*C0)+ z*ratio / (2*C0))

def dV_dC0_analytic(K, Tmat, lam, Cstar, C0, phi):
    lower = max(K, s_min(lam, Cstar, r))

    integral, _ = quad(
        lambda s: (
            (s - K)
           *index_density(s, Tmat, r, lam, Cstar, C0, phi)
           *dp_dC0_factor(s, Tmat, r, lam, Cstar, C0, phi)
        ),
        lower,
        np.inf,
        epsabs=1e-8,
        epsrel=1e-6,
        limit=200
    )

    return np.exp(-r*Tmat)*integral

T_call_13 = 1.0
T_fut_13 = 2.0
K_13 = S0

dVdC0 = dV_dC0_analytic(K_13, T_call_13, lam, Cstar, C0, phi)

dGdC0 = np.exp(-lam*T_fut_13) / (r + lam)

pi0 = dVdC0 / dGdC0

print("dV/dC0 =", dVdC0)
print("dG/dC0 =", dGdC0)
print("Initial number of futures contracts =", pi0)

dV/dC0 = 0.6874764463771533
dG/dC0 = 0.654897931780999
Initial number of futures contracts = 1.0497459421005604


In [5]:
#finite-difference hedge ratio

def model_call_general(K, Tmat, lam, Cstar, C0, phi):
    lower = max(K, s_min(lam, Cstar, r))

    integral, _ = quad(
        lambda s: (s - K)*index_density(s, Tmat, r, lam, Cstar, C0, phi),
        lower,
        np.inf,
        epsabs=1e-8,
        epsrel=1e-6,
        limit=200
    )

    return np.exp(-r*Tmat)*integral


def index_futures_price(Tmat, lam, Cstar, C0):
    return (
        Cstar / r
        + np.exp(-lam*Tmat)*(C0 - Cstar) / (r + lam)
    )


# 1Y ATM call
T_call_13 = 1.0
K_13 = S0

# 2Y index futures
T_fut_13 = 2.0

# finite-difference step
h = 1e-4*C0

V_plus = model_call_general(K_13, T_call_13, lam, Cstar, C0 + h, phi)
V_minus = model_call_general(K_13, T_call_13, lam, Cstar, C0 - h, phi)

dV_dC0 = (V_plus - V_minus) / (2*h)

# analytic derivative of the 2Y futures price with respect to C0
dG_dC0 = np.exp(-lam*T_fut_13) / (r + lam)
# initial number of 2Y futures contracts
n0 = dV_dC0 / dG_dC0

print("dV/dC0 =", dV_dC0)
print("dG/dC0 =", dG_dC0)
print("Initial number of 2Y futures contracts =", n0)

dV/dC0 = 0.6874764461882599
dG/dC0 = 0.654897931780999
Initial number of 2Y futures contracts = 1.0497459418121284
